In [2]:
from google.colab import files
import zipfile
import os

uploaded = files.upload()

zip_name = list(uploaded.keys())[0]

os.makedirs("data/raw/olist", exist_ok=True)

with zipfile.ZipFile(zip_name, "r") as zip_ref:
    zip_ref.extractall("data/raw/olist")

print("Dataset extracted successfully.")
print("\nFiles found:")

for file_name in sorted(os.listdir("data/raw/olist")):
    print(file_name)

Saving archive.zip to archive.zip
Dataset extracted successfully.

Files found:
olist_customers_dataset.csv
olist_geolocation_dataset.csv
olist_order_items_dataset.csv
olist_order_payments_dataset.csv
olist_order_reviews_dataset.csv
olist_orders_dataset.csv
olist_products_dataset.csv
olist_sellers_dataset.csv
product_category_name_translation.csv


In [3]:
!pip -q install duckdb pandas matplotlib seaborn plotly

import duckdb
import pandas as pd
from IPython.display import display

con = duckdb.connect("salespulse.duckdb")

TABLE_FILES = {
    "orders": "olist_orders_dataset.csv",
    "order_items": "olist_order_items_dataset.csv",
    "customers": "olist_customers_dataset.csv",
    "products": "olist_products_dataset.csv",
    "sellers": "olist_sellers_dataset.csv",
    "payments": "olist_order_payments_dataset.csv",
    "reviews": "olist_order_reviews_dataset.csv",
    "category_translation": "product_category_name_translation.csv",
}

for table_name, file_name in TABLE_FILES.items():
    file_path = f"data/raw/olist/{file_name}"

    con.execute(f"""
        CREATE OR REPLACE TABLE {table_name} AS
        SELECT *
        FROM read_csv_auto('{file_path}', header = true);
    """)

    row_count = con.execute(
        f"SELECT COUNT(*) FROM {table_name}"
    ).fetchone()[0]

    print(f"{table_name}: {row_count:,} rows")

print("All tables loaded successfully.")

orders: 99,441 rows
order_items: 112,650 rows
customers: 99,441 rows
products: 32,951 rows
sellers: 3,095 rows
payments: 103,886 rows
reviews: 99,224 rows
category_translation: 71 rows
All tables loaded successfully.


In [4]:
print("Available tables:")
display(con.execute("SHOW TABLES").df())

print("Orders table structure:")
display(con.execute("DESCRIBE orders").df())

print("Order-items table structure:")
display(con.execute("DESCRIBE order_items").df())

print("Sample orders:")
display(
    con.execute("""
        SELECT *
        FROM orders
        LIMIT 5
    """).df()
)

Available tables:


,name
0,category_translation
1,customers
2,order_items
3,orders
4,payments
5,products
6,reviews
7,sellers


Orders table structure:


,column_name,column_type,null,key,default,extra
0,order_id,VARCHAR,YES,None,None,None
1,customer_id,VARCHAR,YES,None,None,None
2,order_status,VARCHAR,YES,None,None,None
3,order_purchase_timestamp,TIMESTAMP,YES,None,None,None
4,order_approved_at,TIMESTAMP,YES,None,None,None
5,order_delivered_carrier_date,TIMESTAMP,YES,None,None,None
6,order_delivered_customer_date,TIMESTAMP,YES,None,None,None
7,order_estimated_delivery_date,TIMESTAMP,YES,None,None,None


Order-items table structure:


,column_name,column_type,null,key,default,extra
0,order_id,VARCHAR,YES,None,None,None
1,order_item_id,BIGINT,YES,None,None,None
2,product_id,VARCHAR,YES,None,None,None
3,seller_id,VARCHAR,YES,None,None,None
4,shipping_limit_date,TIMESTAMP,YES,None,None,None
5,price,DOUBLE,YES,None,None,None
6,freight_value,DOUBLE,YES,None,None,None


Sample orders:


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26


## Entity Relationship Diagram

The database relationships are:

- One customer can place many orders.
- One order can contain many order items.
- One product can appear in many order items.
- One seller can fulfill many order items.
- One order can have many payments.
- One order can have reviews.

```mermaid
erDiagram
    CUSTOMERS ||--o{ ORDERS : places
    ORDERS ||--o{ ORDER_ITEMS : contains
    PRODUCTS ||--o{ ORDER_ITEMS : included_in
    SELLERS ||--o{ ORDER_ITEMS : fulfills
    ORDERS ||--o{ PAYMENTS : has
    ORDERS ||--o{ REVIEWS : receives

    CUSTOMERS {
        string customer_id PK
        string customer_unique_id
        string customer_city
        string customer_state
    }

    ORDERS {
        string order_id PK
        string customer_id FK
        string order_status
        timestamp order_purchase_timestamp
        timestamp order_delivered_customer_date
        timestamp order_estimated_delivery_date
    }

    ORDER_ITEMS {
        string order_id FK
        integer order_item_id
        string product_id FK
        string seller_id FK
        double price
        double freight_value
    }

    PRODUCTS {
        string product_id PK
        string product_category_name
    }

    SELLERS {
        string seller_id PK
        string seller_city
        string seller_state
    }

    PAYMENTS {
        string order_id FK
        string payment_type
        double payment_value
    }

    REVIEWS {
        string review_id PK
        string order_id FK
        integer review_score
    }
```

In [5]:
con.execute("""
CREATE OR REPLACE VIEW fact_sales AS

WITH clean_orders AS (
    SELECT
        order_id,
        customer_id,
        order_status,
        TRY_CAST(order_purchase_timestamp AS TIMESTAMP) AS order_ts,
        TRY_CAST(order_delivered_customer_date AS TIMESTAMP) AS delivered_ts,
        TRY_CAST(order_estimated_delivery_date AS TIMESTAMP) AS estimated_ts
    FROM orders
),

clean_items AS (
    SELECT
        order_id,
        order_item_id,
        product_id,
        seller_id,
        TRY_CAST(price AS DOUBLE) AS item_revenue,
        TRY_CAST(freight_value AS DOUBLE) AS freight_value
    FROM order_items
),

clean_customers AS (
    SELECT
        customer_id,
        customer_unique_id,
        customer_city,
        customer_state
    FROM customers
),

clean_products AS (
    SELECT
        product_id,
        product_category_name
    FROM products
)

SELECT
    o.order_id,
    o.customer_id,
    c.customer_unique_id,
    c.customer_city,
    c.customer_state,
    o.order_status,

    o.order_ts,
    CAST(o.order_ts AS DATE) AS order_date,
    DATE_TRUNC('month', o.order_ts) AS order_month,

    i.order_item_id,
    i.product_id,
    i.seller_id,

    COALESCE(
        t.product_category_name_english,
        p.product_category_name,
        'unknown'
    ) AS category,

    i.item_revenue,
    i.freight_value,
    i.item_revenue + i.freight_value AS item_total_value,

    DATE_DIFF(
        'day',
        CAST(o.order_ts AS DATE),
        CAST(o.delivered_ts AS DATE)
    ) AS delivery_days,

    CASE
        WHEN o.delivered_ts > o.estimated_ts THEN 1
        ELSE 0
    END AS late_flag

FROM clean_orders o
LEFT JOIN clean_customers c
    ON o.customer_id = c.customer_id
LEFT JOIN clean_items i
    ON o.order_id = i.order_id
LEFT JOIN clean_products p
    ON i.product_id = p.product_id
LEFT JOIN category_translation t
    ON p.product_category_name = t.product_category_name
""")

print("fact_sales view created successfully.")

fact_sales view created successfully.


In [6]:
display(
    con.execute("""
        SELECT *
        FROM fact_sales
        LIMIT 10
    """).df()
)

,order_id,customer_id,customer_unique_id,customer_city,customer_state,order_status,order_ts,order_date,order_month,order_item_id,product_id,seller_id,category,item_revenue,freight_value,item_total_value,delivery_days,late_flag
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,7c396fd4830fd04220f754e42b4e5bff,sao paulo,SP,delivered,2017-10-02 10:56:33,2017-10-02,2017-10-01,1,87285b34884572647811a353c7ac498a,3504c0cb71d7fa48d967e0e4c94d59d9,housewares,29.99,8.72,38.71,8,0
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,af07308b275d755c9edb36a90c618231,barreiras,BA,delivered,2018-07-24 20:41:37,2018-07-24,2018-07-01,1,595fac2a385ac33a80bd5114aec74eb8,289cdb325fb7e7f891c38608bf9e0962,perfumery,118.70,22.76,141.46,14,0
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,3a653a41f6f9fc3d2a113cf8398680e8,vianopolis,GO,delivered,2018-08-08 08:38:49,2018-08-08,2018-08-01,1,aa4383b373c6aca5d8797843e5594415,4869f7a5dfa277a7dca6462dcf3b52b2,auto,159.90,19.22,179.12,9,0
3,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,72632f0f9dd73dfee390c9b22eb56dd6,santo andre,SP,delivered,2018-02-13 21:18:39,2018-02-13,2018-02-01,1,65266b2da20d04dbe00c5c2d3bb7859e,2c9e548be18521d1c43cde1c582c6de8,stationery,19.90,8.72,28.62,3,0
4,a4591c265e18cb1dcee52889e2d8acc3,503740e9ca751ccdda7ba28e9ab8f608,80bb27c7c16e8f973207a5086ab329e2,congonhinhas,PR,delivered,2017-07-09 21:57:05,2017-07-09,2017-07-01,1,060cb19345d90064d1015407193c233d,8581055ce74af1daba164fdbd55a40de,auto,147.90,27.36,175.26,17,0
5,6514b8ad8028c9f2cc2374ded245783f,9bdf08b4b3b52b5526ff42d37d47f222,932afa1e708222e5821dac9cd5db4cae,nilopolis,RJ,delivered,2017-05-16 13:10:30,2017-05-16,2017-05-01,1,4520766ec412348b8d4caa5e8a18c464,16090f2ca825584b5a147ab24aa30c86,auto,59.99,15.17,75.16,10,0
6,76c6e866289321a7c93b82b54852dc33,f54a9f0e6b351c431402b8461ea51999,39382392765b6dc74812866ee5ee92a7,faxinalzinho,RS,delivered,2017-01-23 18:29:09,2017-01-23,2017-01-01,1,ac1789e492dcd698c5c10b97a671243a,63b9ae557efed31d1f7687917d248a8d,furniture_decor,19.90,16.05,35.95,10,0
7,e69bfb5eb88e0ed6a785585b27e16dbf,31ad1d1b63eb9962463f764d4e6e0c9d,299905e3934e9e181bfb2e164dd4b4f8,sorocaba,SP,delivered,2017-07-29 11:55:02,2017-07-29,2017-07-01,1,9a78fb9862b10749a117f7fc3c31f051,7c67e1448b00f6e969d365cea6b010ab,office_furniture,149.99,19.77,169.76,18,0
8,e6ce16cb79ec1d90b1da9085a6118aeb,494dded5b201313c64ed7f100595b95c,f2a85dec752b8517b5e58a06ff3cd937,rio de janeiro,RJ,delivered,2017-05-16 19:41:10,2017-05-16,2017-05-01,2,08574b074924071f4e201e151b152b4e,001cca7ae9ae17fb1caed9dfb1094831,garden_tools,99.00,30.53,129.53,13,0
9,34513ce0c4fab462a55830c0989c7edb,7711cf624183d843aafe81855097bc37,782987b81c92239d922aa49d6bd4200b,sao paulo,SP,delivered,2017-07-13 19:58:11,2017-07-13,2017-07-01,1,f7e0fa615b386bc9a8b9eb52bc1fff76,87142160b41353c4e5fca2360caf6f92,computers_accessories,98.00,16.13,114.13,6,0


In [7]:
con.execute("""
CREATE OR REPLACE VIEW order_summary AS

SELECT
    order_id,
    MAX(customer_unique_id) AS customer_unique_id,
    MAX(customer_state) AS customer_state,
    MAX(order_status) AS order_status,
    MIN(order_date) AS order_date,

    SUM(item_revenue) AS order_revenue,
    SUM(freight_value) AS total_freight,
    COUNT(*) AS item_count,

    MAX(delivery_days) AS delivery_days,
    MAX(late_flag) AS late_flag

FROM fact_sales
GROUP BY order_id
""")

print("order_summary view created successfully.")

order_summary view created successfully.


In [8]:
display(
    con.execute("""
        SELECT *
        FROM order_summary
        LIMIT 10
    """).df()
)

,order_id,customer_unique_id,customer_state,order_status,order_date,order_revenue,total_freight,item_count,delivery_days,late_flag
0,e481f51cbdc54678b7cc49136f2d6af7,7c396fd4830fd04220f754e42b4e5bff,SP,delivered,2017-10-02,29.99,8.72,1,8,0
1,47770eb9100c2d0c44946d9cf07ec65d,3a653a41f6f9fc3d2a113cf8398680e8,GO,delivered,2018-08-08,159.90,19.22,1,9,0
2,ad21c59c0840e6cb83a9ceb5573f8159,72632f0f9dd73dfee390c9b22eb56dd6,SP,delivered,2018-02-13,19.90,8.72,1,3,0
3,a4591c265e18cb1dcee52889e2d8acc3,80bb27c7c16e8f973207a5086ab329e2,PR,delivered,2017-07-09,147.90,27.36,1,17,0
4,76c6e866289321a7c93b82b54852dc33,39382392765b6dc74812866ee5ee92a7,RS,delivered,2017-01-23,19.90,16.05,1,10,0
5,82566a660a982b15fb86e904c8d32918,e97109680b052ee858d93a539597bba7,MG,delivered,2018-06-07,31.90,18.23,1,12,0
6,5ff96c15d0b717ac6ad1f3d77225a350,e2dfa3127fedbbca9707b36304996dab,SP,delivered,2018-07-25,19.90,12.80,1,5,0
7,432aaf21d85167c2c86ec9448c4e42cc,04cf8185c71090d28baa4407b2e6d600,SP,delivered,2018-03-01,38.25,16.11,1,11,0
8,dcb36b511fcac050b97cd5c05de84dc3,ccafc1c3f270410521c3c6f3b249870f,GO,delivered,2018-06-07,132.40,14.05,1,14,0
9,403b97836b0c04a622354cf531062e5f,6e26bbeaa107ec34112c64e1ee31c0f5,RJ,delivered,2018-01-02,1299.00,77.45,1,18,0


In [9]:
validation = con.execute("""
SELECT
    (SELECT COUNT(*) FROM orders) AS raw_order_rows,

    (SELECT COUNT(*) FROM order_summary) AS summary_order_rows,

    (SELECT COUNT(DISTINCT order_id) FROM fact_sales)
        AS fact_distinct_orders,

    (SELECT COUNT(*) FROM fact_sales)
        AS fact_item_rows,

    (SELECT SUM(order_revenue) FROM order_summary)
        AS order_summary_revenue,

    (SELECT SUM(item_revenue) FROM fact_sales)
        AS fact_item_revenue
""").df()

display(validation)

,raw_order_rows,summary_order_rows,fact_distinct_orders,fact_item_rows,order_summary_revenue,fact_item_revenue
0,99441,99441,99441,113425,1.359164e+07,1.359164e+07


In [10]:
duplicate_orders = con.execute("""
SELECT
    order_id,
    COUNT(*) AS row_count
FROM order_summary
GROUP BY order_id
HAVING COUNT(*) > 1
""").df()

print("Duplicate order IDs:", len(duplicate_orders))
display(duplicate_orders.head())

Duplicate order IDs: 0


,order_id,row_count


In [11]:
# Q01: What is the total revenue?
display(
    con.execute("""
        -- Business question: What is the total product revenue?
        SELECT
            ROUND(SUM(item_revenue), 2) AS total_revenue
        FROM fact_sales
    """).df()
)

# Q02: How many orders and customers do we have?
display(
    con.execute("""
        -- Business question: How many orders and unique customers exist?
        SELECT
            COUNT(*) AS total_orders,
            COUNT(DISTINCT customer_unique_id) AS unique_customers
        FROM order_summary
    """).df()
)

# Q03: What is the average order value?
display(
    con.execute("""
        -- Business question: What is the average revenue per order?
        SELECT
            ROUND(AVG(order_revenue), 2) AS average_order_value
        FROM order_summary
    """).df()
)

# Q04: What is monthly revenue?
display(
    con.execute("""
        -- Business question: How does revenue change month by month?
        SELECT
            order_month,
            ROUND(SUM(item_revenue), 2) AS monthly_revenue
        FROM fact_sales
        GROUP BY order_month
        ORDER BY order_month
    """).df()
)

# Q05: Which product categories generate the most revenue?
display(
    con.execute("""
        -- Business question: Which product categories generate the most revenue?
        SELECT
            category,
            ROUND(SUM(item_revenue), 2) AS category_revenue,
            COUNT(DISTINCT order_id) AS order_count
        FROM fact_sales
        GROUP BY category
        ORDER BY category_revenue DESC
        LIMIT 15
    """).df()
)

,total_revenue
0,13591643.7


,total_orders,unique_customers
0,99441,96096


,average_order_value
0,137.75


,order_month,monthly_revenue
0,2016-09-01,267.36
1,2016-10-01,49507.66
2,2016-12-01,10.90
3,2017-01-01,120312.87
4,2017-02-01,247303.02
5,2017-03-01,374344.30
6,2017-04-01,359927.23
7,2017-05-01,506071.14
8,2017-06-01,433038.60
9,2017-07-01,498031.48


,category,category_revenue,order_count
0,health_beauty,1258681.34,8836
1,watches_gifts,1205005.68,5624
2,bed_bath_table,1036988.68,9417
3,sports_leisure,988048.97,7720
4,computers_accessories,911954.32,6689
5,furniture_decor,729762.49,6449
6,cool_stuff,635290.85,3632
7,housewares,632248.66,5884
8,auto,592720.11,3897
9,garden_tools,485256.46,3518


In [12]:
# Q06: Which customer states generate the most revenue?
display(
    con.execute("""
        -- Business question: Which states generate the most revenue?
        SELECT
            customer_state,
            ROUND(SUM(order_revenue), 2) AS total_revenue,
            COUNT(*) AS order_count
        FROM order_summary
        GROUP BY customer_state
        ORDER BY total_revenue DESC
        LIMIT 15
    """).df()
)

# Q07: What are the most common order statuses?
display(
    con.execute("""
        -- Business question: How many orders exist in each status?
        SELECT
            order_status,
            COUNT(*) AS order_count
        FROM order_summary
        GROUP BY order_status
        ORDER BY order_count DESC
    """).df()
)

# Q08: Which payment methods are used most often?
display(
    con.execute("""
        -- Business question: Which payment methods are most common?
        SELECT
            payment_type,
            COUNT(DISTINCT order_id) AS order_count,
            ROUND(SUM(TRY_CAST(payment_value AS DOUBLE)), 2)
                AS payment_total
        FROM payments
        GROUP BY payment_type
        ORDER BY order_count DESC
    """).df()
)

# Q09: What is the average review score?
display(
    con.execute("""
        -- Business question: What is the average customer review score?
        SELECT
            ROUND(AVG(TRY_CAST(review_score AS DOUBLE)), 2)
                AS average_review_score,
            COUNT(*) AS review_count
        FROM reviews
    """).df()
)

# Q10: Which sellers generate the most revenue?
display(
    con.execute("""
        -- Business question: Which sellers generate the most revenue?
        SELECT
            seller_id,
            ROUND(SUM(item_revenue), 2) AS seller_revenue,
            COUNT(DISTINCT order_id) AS order_count
        FROM fact_sales
        GROUP BY seller_id
        ORDER BY seller_revenue DESC
        LIMIT 15
    """).df()
)

,customer_state,total_revenue,order_count
0,SP,5202955.05,41746
1,RJ,1824092.67,12852
2,MG,1585308.03,11635
3,RS,750304.02,5466
4,PR,683083.76,5045
5,SC,520553.34,3637
6,BA,511349.99,3380
7,DF,302603.94,2140
8,GO,294591.95,2020
9,ES,275037.31,2033


,order_status,order_count
0,delivered,96478
1,shipped,1107
2,canceled,625
3,unavailable,609
4,invoiced,314
5,processing,301
6,created,5
7,approved,2


,payment_type,order_count,payment_total
0,credit_card,76505,12542084.19
1,boleto,19784,2869361.27
2,voucher,3866,379436.87
3,debit_card,1528,217989.79
4,not_defined,3,0.00


,average_review_score,review_count
0,4.09,99224


,seller_id,seller_revenue,order_count
0,4869f7a5dfa277a7dca6462dcf3b52b2,229472.63,1132
1,53243585a1d6dc2643021fd1853d8905,222776.05,358
2,4a3ca9315b744ce9f8e9374361493884,200472.92,1806
3,fa1c13f2614d7b5c4749cbc52fecda94,194042.03,585
4,7c67e1448b00f6e969d365cea6b010ab,187923.89,982
5,7e93a43ef30c4f03f38b393420bc753a,176431.87,336
6,da8622b14eb17ae2831f4ac5b9dab84a,160236.57,1314
7,7a67c85e85bb2ce8582c35f2203ad736,141745.53,1160
8,1025f0e2d44d7041d6cf58b6550e0bfa,138968.55,915
9,955fee9216a65b617aa5c0531780ce60,135171.70,1287


In [13]:
# Q11: What is each customer's lifetime revenue?
display(
    con.execute("""
        -- Business question: Which customers generate the most lifetime revenue?
        WITH customer_metrics AS (
            SELECT
                customer_unique_id,
                COUNT(*) AS order_count,
                ROUND(SUM(order_revenue), 2) AS lifetime_revenue,
                MIN(order_date) AS first_order_date,
                MAX(order_date) AS last_order_date
            FROM order_summary
            GROUP BY customer_unique_id
        )
        SELECT *
        FROM customer_metrics
        ORDER BY lifetime_revenue DESC
        LIMIT 20
    """).df()
)

# Q12: When did each customer place their first order?
display(
    con.execute("""
        -- Business question: When did each customer first purchase?
        WITH first_orders AS (
            SELECT
                customer_unique_id,
                MIN(order_date) AS first_order_date
            FROM order_summary
            GROUP BY customer_unique_id
        )
        SELECT *
        FROM first_orders
        ORDER BY first_order_date
        LIMIT 20
    """).df()
)

# Q13: What percentage of customers are repeat customers?
display(
    con.execute("""
        -- Business question: What is the repeat-customer rate?
        WITH customer_order_counts AS (
            SELECT
                customer_unique_id,
                COUNT(*) AS order_count
            FROM order_summary
            GROUP BY customer_unique_id
        )
        SELECT
            COUNT(*) AS total_customers,
            SUM(
                CASE WHEN order_count > 1 THEN 1 ELSE 0 END
            ) AS repeat_customers,
            ROUND(
                100.0 * SUM(
                    CASE WHEN order_count > 1 THEN 1 ELSE 0 END
                ) / COUNT(*),
                2
            ) AS repeat_customer_percent
        FROM customer_order_counts
    """).df()
)

# Q14: What is the average delivery time?
display(
    con.execute("""
        -- Business question: How many days does delivery take on average?
        WITH delivered_orders AS (
            SELECT
                order_id,
                delivery_days
            FROM order_summary
            WHERE order_status = 'delivered'
              AND delivery_days IS NOT NULL
        )
        SELECT
            COUNT(*) AS delivered_orders,
            ROUND(AVG(delivery_days), 2) AS average_delivery_days,
            MIN(delivery_days) AS fastest_delivery_days,
            MAX(delivery_days) AS slowest_delivery_days
        FROM delivered_orders
    """).df()
)

# Q15: Which states have the highest late-delivery rate?
display(
    con.execute("""
        -- Business question: Which states have the highest late-delivery rate?
        WITH state_delivery AS (
            SELECT
                customer_state,
                COUNT(*) AS delivered_orders,
                SUM(late_flag) AS late_orders
            FROM order_summary
            WHERE order_status = 'delivered'
              AND delivery_days IS NOT NULL
            GROUP BY customer_state
        )
        SELECT
            customer_state,
            delivered_orders,
            late_orders,
            ROUND(
                100.0 * late_orders / delivered_orders,
                2
            ) AS late_delivery_percent
        FROM state_delivery
        WHERE delivered_orders >= 100
        ORDER BY late_delivery_percent DESC
    """).df()
)

,customer_unique_id,order_count,lifetime_revenue,first_order_date,last_order_date
0,0a0a92112bd4c708ca5fde585afaa872,1,13440.00,2017-09-29,2017-09-29
1,da122df9eeddfedc1dc1f5349a1a690c,2,7388.00,2017-04-01,2017-04-01
2,763c8b1c9c68a0229c42c9fc6f662b93,1,7160.00,2018-07-15,2018-07-15
3,dc4802a71eae9be1dd28f5d788ceb526,1,6735.00,2017-02-12,2017-02-12
4,459bef486812aa25204be022145caa62,1,6729.00,2018-07-25,2018-07-25
5,ff4159b92c40ebe40454e3e6a7c35ed6,1,6499.00,2017-05-24,2017-05-24
6,4007669dec559734d6f53e029e360987,1,5934.60,2017-11-24,2017-11-24
7,eebb5dda148d3893cdaf5b5ca3040ccb,1,4690.00,2017-04-18,2017-04-18
8,5d0a2980b292d049061542014e8960bf,1,4599.90,2018-07-12,2018-07-12
9,48e1ac109decbb87765a3eade6854098,1,4590.00,2018-06-22,2018-06-22


,customer_unique_id,first_order_date
0,b7d76e111c89f7ebf14761390f0f7d17,2016-09-04
1,4854e9b3feff728c13ee5fc7d1547e92,2016-09-05
2,009b0127b727ab0ba422f6d9604487c7,2016-09-13
3,830d5b7aaa3b6f1e9ad63703bec97d23,2016-09-15
4,0eb1ee9dba87f5b36b4613a65074337c,2016-10-02
5,2f64e403852e6893ae37485d5fcacdaf,2016-10-03
6,32ea3bdedab835c3aa6cb68ce66565ef,2016-10-03
7,87776adb449c551e74c13fc34f036105,2016-10-03
8,8d3a54507421dbd2ce0a1d58046826e0,2016-10-03
9,61db744d2f835035a5625b59350c6b63,2016-10-03


,total_customers,repeat_customers,repeat_customer_percent
0,96096,2997.0,3.12


,delivered_orders,average_delivery_days,fastest_delivery_days,slowest_delivery_days
0,96470,12.5,0,210


,customer_state,delivered_orders,late_orders,late_delivery_percent
0,AL,397,95.0,23.93
1,MA,717,141.0,19.67
2,PI,476,76.0,15.97
3,CE,1279,196.0,15.32
4,SE,335,51.0,15.22
5,BA,3256,457.0,14.04
6,RJ,12350,1664.0,13.47
7,TO,274,35.0,12.77
8,PA,946,117.0,12.37
9,ES,1995,244.0,12.23


In [14]:
# Q16: Which categories have the highest freight costs?
display(
    con.execute("""
        -- Business question: Which categories have the highest freight cost?
        WITH category_freight AS (
            SELECT
                category,
                SUM(freight_value) AS total_freight,
                SUM(item_revenue) AS total_revenue,
                COUNT(DISTINCT order_id) AS order_count
            FROM fact_sales
            GROUP BY category
        )
        SELECT
            category,
            ROUND(total_freight, 2) AS total_freight,
            ROUND(total_revenue, 2) AS total_revenue,
            order_count,
            ROUND(
                100.0 * total_freight / NULLIF(total_revenue, 0),
                2
            ) AS freight_as_percent_of_revenue
        FROM category_freight
        ORDER BY total_freight DESC
        LIMIT 15
    """).df()
)

# Q17: Which seller states generate the most revenue?
display(
    con.execute("""
        -- Business question: Which seller states generate the most revenue?
        WITH seller_state_revenue AS (
            SELECT
                s.seller_state,
                SUM(fs.item_revenue) AS revenue,
                COUNT(DISTINCT fs.order_id) AS order_count
            FROM fact_sales fs
            LEFT JOIN sellers s
                ON fs.seller_id = s.seller_id
            GROUP BY s.seller_state
        )
        SELECT
            seller_state,
            ROUND(revenue, 2) AS revenue,
            order_count
        FROM seller_state_revenue
        ORDER BY revenue DESC
    """).df()
)

# Q18: How many customers and orders were active each month?
display(
    con.execute("""
        -- Business question: How do monthly customers and orders change?
        WITH monthly_activity AS (
            SELECT
                order_month,
                COUNT(DISTINCT order_id) AS order_count,
                COUNT(DISTINCT customer_unique_id) AS customer_count,
                SUM(item_revenue) AS revenue
            FROM fact_sales
            GROUP BY order_month
        )
        SELECT
            order_month,
            order_count,
            customer_count,
            ROUND(revenue, 2) AS revenue
        FROM monthly_activity
        ORDER BY order_month
    """).df()
)

# Q19: What revenue comes from each order status?
display(
    con.execute("""
        -- Business question: How much revenue is associated with each order status?
        WITH status_metrics AS (
            SELECT
                order_status,
                COUNT(*) AS order_count,
                SUM(order_revenue) AS revenue
            FROM order_summary
            GROUP BY order_status
        )
        SELECT
            order_status,
            order_count,
            ROUND(revenue, 2) AS revenue
        FROM status_metrics
        ORDER BY revenue DESC
    """).df()
)

# Q20: Which customer states have the best review scores?
display(
    con.execute("""
        -- Business question: Which customer states have the highest review scores?
        WITH review_by_order AS (
            SELECT
                order_id,
                AVG(TRY_CAST(review_score AS DOUBLE)) AS review_score
            FROM reviews
            GROUP BY order_id
        ),

        state_reviews AS (
            SELECT
                os.customer_state,
                AVG(r.review_score) AS average_review_score,
                COUNT(*) AS reviewed_orders
            FROM order_summary os
            INNER JOIN review_by_order r
                ON os.order_id = r.order_id
            GROUP BY os.customer_state
        )

        SELECT
            customer_state,
            ROUND(average_review_score, 2) AS average_review_score,
            reviewed_orders
        FROM state_reviews
        WHERE reviewed_orders >= 100
        ORDER BY average_review_score DESC
    """).df()
)

,category,total_freight,total_revenue,order_count,freight_as_percent_of_revenue
0,bed_bath_table,204693.04,1036988.68,9417,19.74
1,health_beauty,182566.73,1258681.34,8836,14.50
2,furniture_decor,172749.30,729762.49,6449,23.67
3,sports_leisure,168607.51,988048.97,7720,17.06
4,computers_accessories,147318.08,911954.32,6689,16.15
5,housewares,146149.11,632248.66,5884,23.12
6,watches_gifts,100535.93,1205005.68,5624,8.34
7,garden_tools,98962.75,485256.46,3518,20.39
8,auto,92664.21,592720.11,3897,15.63
9,cool_stuff,84039.10,635290.85,3632,13.23


,seller_state,revenue,order_count
0,SP,8753396.21,70188
1,PR,1261887.21,7673
2,MG,1011564.74,7930
3,RJ,843984.22,4353
4,SC,632426.07,3667
5,RS,378559.54,1989
6,BA,285561.56,569
7,DF,97749.48,824
8,PE,91493.85,406
9,GO,66399.21,463


,order_month,order_count,customer_count,revenue
0,2016-09-01,4,4,267.36
1,2016-10-01,324,321,49507.66
2,2016-12-01,1,1,10.90
3,2017-01-01,800,765,120312.87
4,2017-02-01,1780,1755,247303.02
5,2017-03-01,2682,2642,374344.30
6,2017-04-01,2404,2372,359927.23
7,2017-05-01,3700,3625,506071.14
8,2017-06-01,3245,3180,433038.60
9,2017-07-01,4026,3947,498031.48


,order_status,order_count,revenue
0,delivered,96478,13221498.11
1,shipped,1107,150727.44
2,canceled,625,95235.27
3,invoiced,314,61526.37
4,processing,301,60439.22
5,unavailable,609,2007.69
6,approved,2,209.60
7,created,5,NaN


,customer_state,average_review_score,reviewed_orders
0,AM,4.21,146
1,PR,4.18,5019
2,SP,4.17,41472
3,MG,4.14,11554
4,RS,4.13,5443
5,MS,4.11,713
6,RN,4.11,480
7,MT,4.10,900
8,TO,4.10,279
9,DF,4.07,2128


In [15]:
# Q21: What is month-over-month revenue growth?
display(
    con.execute("""
        -- Business question: How did revenue change from the previous month?
        WITH monthly_revenue AS (
            SELECT
                order_month,
                SUM(item_revenue) AS revenue
            FROM fact_sales
            GROUP BY order_month
        ),

        previous_month AS (
            SELECT
                order_month,
                revenue,
                LAG(revenue) OVER (
                    ORDER BY order_month
                ) AS previous_revenue
            FROM monthly_revenue
        )

        SELECT
            order_month,
            ROUND(revenue, 2) AS revenue,
            ROUND(previous_revenue, 2) AS previous_revenue,
            ROUND(
                100.0 * (revenue - previous_revenue)
                / NULLIF(previous_revenue, 0),
                2
            ) AS mom_growth_percent
        FROM previous_month
        ORDER BY order_month
    """).df()
)

# Q22: What is the cumulative revenue over time?
display(
    con.execute("""
        -- Business question: What is the running total of revenue?
        WITH monthly_revenue AS (
            SELECT
                order_month,
                SUM(item_revenue) AS revenue
            FROM fact_sales
            GROUP BY order_month
        )

        SELECT
            order_month,
            ROUND(revenue, 2) AS revenue,
            ROUND(
                SUM(revenue) OVER (
                    ORDER BY order_month
                    ROWS BETWEEN UNBOUNDED PRECEDING
                    AND CURRENT ROW
                ),
                2
            ) AS running_revenue
        FROM monthly_revenue
        ORDER BY order_month
    """).df()
)

# Q23: What are the top three categories in each month?
display(
    con.execute("""
        -- Business question: Which three categories led revenue each month?
        WITH monthly_category_revenue AS (
            SELECT
                order_month,
                category,
                SUM(item_revenue) AS revenue
            FROM fact_sales
            GROUP BY order_month, category
        ),

        ranked_categories AS (
            SELECT
                order_month,
                category,
                revenue,
                ROW_NUMBER() OVER (
                    PARTITION BY order_month
                    ORDER BY revenue DESC
                ) AS category_rank
            FROM monthly_category_revenue
        )

        SELECT
            order_month,
            category,
            ROUND(revenue, 2) AS revenue,
            category_rank
        FROM ranked_categories
        WHERE category_rank <= 3
        ORDER BY order_month, category_rank
    """).df()
)

# Q24: What is the product rank within each category?
display(
    con.execute("""
        -- Business question: Which products rank highest within each category?
        WITH product_revenue AS (
            SELECT
                category,
                product_id,
                SUM(item_revenue) AS revenue
            FROM fact_sales
            GROUP BY category, product_id
        )

        SELECT
            category,
            product_id,
            ROUND(revenue, 2) AS revenue,
            RANK() OVER (
                PARTITION BY category
                ORDER BY revenue DESC
            ) AS product_rank
        FROM product_revenue
        ORDER BY category, product_rank
    """).df()
)

# Q25: What is the three-month moving-average revenue?
display(
    con.execute("""
        -- Business question: What is the smoothed three-month revenue trend?
        WITH monthly_revenue AS (
            SELECT
                order_month,
                SUM(item_revenue) AS revenue
            FROM fact_sales
            GROUP BY order_month
        )

        SELECT
            order_month,
            ROUND(revenue, 2) AS revenue,
            ROUND(
                AVG(revenue) OVER (
                    ORDER BY order_month
                    ROWS BETWEEN 2 PRECEDING AND CURRENT ROW
                ),
                2
            ) AS three_month_moving_average
        FROM monthly_revenue
        ORDER BY order_month
    """).df()
)

# Q26: What is each customer's order sequence?
display(
    con.execute("""
        -- Business question: Which order number was each customer placing?
        SELECT
            customer_unique_id,
            order_id,
            order_date,
            ROW_NUMBER() OVER (
                PARTITION BY customer_unique_id
                ORDER BY order_date
            ) AS customer_order_number
        FROM order_summary
        ORDER BY customer_unique_id, order_date
        LIMIT 100
    """).df()
)

# Q27: How many days passed between customer orders?
display(
    con.execute("""
        -- Business question: How long did customers wait between orders?
        WITH customer_orders AS (
            SELECT
                customer_unique_id,
                order_id,
                order_date,
                LAG(order_date) OVER (
                    PARTITION BY customer_unique_id
                    ORDER BY order_date
                ) AS previous_order_date
            FROM order_summary
        )

        SELECT
            customer_unique_id,
            order_id,
            order_date,
            previous_order_date,
            DATE_DIFF(
                'day',
                previous_order_date,
                order_date
            ) AS days_between_orders
        FROM customer_orders
        WHERE previous_order_date IS NOT NULL
        ORDER BY customer_unique_id, order_date
        LIMIT 100
    """).df()
)

,order_month,revenue,previous_revenue,mom_growth_percent
0,2016-09-01,267.36,NaN,NaN
1,2016-10-01,49507.66,267.36,18417.23
2,2016-12-01,10.90,49507.66,-99.98
3,2017-01-01,120312.87,10.90,1103687.80
4,2017-02-01,247303.02,120312.87,105.55
5,2017-03-01,374344.30,247303.02,51.37
6,2017-04-01,359927.23,374344.30,-3.85
7,2017-05-01,506071.14,359927.23,40.60
8,2017-06-01,433038.60,506071.14,-14.43
9,2017-07-01,498031.48,433038.60,15.01


,order_month,revenue,running_revenue
0,2016-09-01,267.36,267.36
1,2016-10-01,49507.66,49775.02
2,2016-12-01,10.90,49785.92
3,2017-01-01,120312.87,170098.79
4,2017-02-01,247303.02,417401.81
5,2017-03-01,374344.30,791746.11
6,2017-04-01,359927.23,1151673.34
7,2017-05-01,506071.14,1657744.48
8,2017-06-01,433038.60,2090783.08
9,2017-07-01,498031.48,2588814.56


,order_month,category,revenue,category_rank
0,2016-09-01,health_beauty,134.97,1
1,2016-09-01,furniture_decor,72.89,2
2,2016-09-01,telephony,59.50,3
3,2016-10-01,furniture_decor,5807.89,1
4,2016-10-01,perfumery,5688.70,2
...,...,...,...,...
65,2018-08-01,watches_gifts,72277.06,2
66,2018-08-01,bed_bath_table,61035.88,3
67,2018-09-01,kitchen_dining_laundry_garden_furniture,145.00,1
68,2018-09-01,unknown,NaN,2


,category,product_id,revenue,product_rank
0,agro_industry_and_commerce,11250b0d4b709fee92441c5f34122aed,9111.00,1
1,agro_industry_and_commerce,423a6644f0aa529e8828ff1f91003690,8043.00,2
2,agro_industry_and_commerce,672e757f331900b9deea127a2a7b79fd,6885.00,3
3,agro_industry_and_commerce,c183fd5d2abf05873fa6e1014ed9e06c,5934.60,4
4,agro_industry_and_commerce,2b69866f22de8dad69c976771daba91c,2990.00,5
...,...,...,...,...
32947,watches_gifts,e27a738cbbcf6c8878c713af56835a19,23.90,1325
32948,watches_gifts,22dcd61aae0f6d1acc992914bc67a0e5,19.90,1326
32949,watches_gifts,737ee9efa4782433c69fb8d80e925dde,19.90,1326
32950,watches_gifts,f194a56fe15513535bd5b51e2f7086c3,10.99,1328


,order_month,revenue,three_month_moving_average
0,2016-09-01,267.36,267.36
1,2016-10-01,49507.66,24887.51
2,2016-12-01,10.90,16595.31
3,2017-01-01,120312.87,56610.48
4,2017-02-01,247303.02,122542.26
5,2017-03-01,374344.30,247320.06
6,2017-04-01,359927.23,327191.52
7,2017-05-01,506071.14,413447.56
8,2017-06-01,433038.60,433012.32
9,2017-07-01,498031.48,479047.07


,customer_unique_id,order_id,order_date,customer_order_number
0,0000366f3b9a7992bf8c76cfdf3221e2,e22acc9c116caa3f2b7121bbb380d08e,2018-05-10,1
1,0000b849f77a49e4a4ce2b2a4ca5be3f,3594e05a005ac4d06a72673270ef9ec9,2018-05-07,1
2,0000f46a3911fa3c0805444483337064,b33ec3b699337181488304f362a6b734,2017-03-10,1
3,0000f6ccb0745a6a4b88665a16c9f078,41272756ecddd9a9ed0180413cc22fb6,2017-10-12,1
4,0004aac84e0df4da2b147fca70cf8255,d957021f1127559cd947b62533f484f7,2017-11-14,1
...,...,...,...,...
95,003ae409f37c3c30cb1c974af3a42692,c0c2ece9e641c6fc11690ceafbc7469e,2018-08-15,1
96,003c2067070f5b3d923235b1ea1cc17b,a250e33c3d40fa9dd726391ebc5a4abb,2018-08-06,1
97,003c457932a554e4df13834237b1ed00,081081d7c1bab4679737b9254c3e992a,2018-01-05,1
98,003d56767e53e08671de00da3fba8d40,7965b290cc7fe4790d45eaeded895967,2017-12-01,1


,customer_unique_id,order_id,order_date,previous_order_date,days_between_orders
0,00172711b30d52eea8b313a7f2cced02,c306eca42d32507b970739b5b6a5a33a,2018-08-13,2018-07-28,16
1,004288347e5e88a27ded2bb23747066c,08204559bebd39e09ee52dcb56d8faa2,2018-01-14,2017-07-27,171
2,004b45ec5c64187465168251cd1c9c2f,9392c5e72885ad5aba87e6223ca9838d,2018-05-26,2017-09-01,267
3,0058f300f57d7b93c477a131a59b36c3,81a93b2fa39e104b865b2bc471c16008,2018-03-22,2018-02-19,31
4,00a39521eb40f7012db50455bf083460,cea3e6c11eb60acb9d8d4d51694832f8,2018-06-03,2018-05-23,11
...,...,...,...,...,...
95,083ca1aa470c280236380973a48f77c6,96dad13c399dce9663d3bd03e84cea2b,2017-08-16,2017-08-05,11
96,083ca1aa470c280236380973a48f77c6,f79d2206d0301d47bc2a623636ea53d1,2018-08-13,2017-08-16,362
97,083ce299ea3fc1eba79bd0b17f64555b,1342ab50383d7471a3bfd75ca93d38d6,2018-08-23,2018-01-22,213
98,084be429c23e342acf5084ac6156d22e,b461e43868116fb81d1d91959ae5ea65,2018-07-22,2018-04-24,89


In [16]:
%%writefile salespulse_queries.sql

-- SalesPulse SQL Query Library
-- Database views required:
-- fact_sales and order_summary

-- Q01: What is total revenue?
SELECT ROUND(SUM(item_revenue), 2) AS total_revenue
FROM fact_sales;


-- Q02: How many orders and unique customers exist?
SELECT
    COUNT(*) AS total_orders,
    COUNT(DISTINCT customer_unique_id) AS unique_customers
FROM order_summary;


-- Q03: What is the average order value?
SELECT ROUND(AVG(order_revenue), 2) AS average_order_value
FROM order_summary;


-- Q04: What is monthly revenue?
SELECT
    order_month,
    ROUND(SUM(item_revenue), 2) AS monthly_revenue
FROM fact_sales
GROUP BY order_month
ORDER BY order_month;


-- Q05: Which product categories generate the most revenue?
SELECT
    category,
    ROUND(SUM(item_revenue), 2) AS category_revenue,
    COUNT(DISTINCT order_id) AS order_count
FROM fact_sales
GROUP BY category
ORDER BY category_revenue DESC
LIMIT 15;


-- Q06: Which customer states generate the most revenue?
SELECT
    customer_state,
    ROUND(SUM(order_revenue), 2) AS total_revenue,
    COUNT(*) AS order_count
FROM order_summary
GROUP BY customer_state
ORDER BY total_revenue DESC;


-- Q07: How many orders exist in each status?
SELECT
    order_status,
    COUNT(*) AS order_count
FROM order_summary
GROUP BY order_status
ORDER BY order_count DESC;


-- Q08: Which payment methods are most common?
SELECT
    payment_type,
    COUNT(DISTINCT order_id) AS order_count,
    ROUND(SUM(TRY_CAST(payment_value AS DOUBLE)), 2) AS payment_total
FROM payments
GROUP BY payment_type
ORDER BY order_count DESC;


-- Q09: What is the average customer review score?
SELECT
    ROUND(AVG(TRY_CAST(review_score AS DOUBLE)), 2) AS average_review_score,
    COUNT(*) AS review_count
FROM reviews;


-- Q10: Which sellers generate the most revenue?
SELECT
    seller_id,
    ROUND(SUM(item_revenue), 2) AS seller_revenue,
    COUNT(DISTINCT order_id) AS order_count
FROM fact_sales
GROUP BY seller_id
ORDER BY seller_revenue DESC
LIMIT 15;


-- Q11: Which customers generate the most lifetime revenue?
WITH customer_metrics AS (
    SELECT
        customer_unique_id,
        COUNT(*) AS order_count,
        SUM(order_revenue) AS lifetime_revenue,
        MIN(order_date) AS first_order_date,
        MAX(order_date) AS last_order_date
    FROM order_summary
    GROUP BY customer_unique_id
)
SELECT *
FROM customer_metrics
ORDER BY lifetime_revenue DESC
LIMIT 20;


-- Q12: When did each customer first purchase?
WITH first_orders AS (
    SELECT
        customer_unique_id,
        MIN(order_date) AS first_order_date
    FROM order_summary
    GROUP BY customer_unique_id
)
SELECT *
FROM first_orders
ORDER BY first_order_date;


-- Q13: What is the repeat-customer rate?
WITH customer_order_counts AS (
    SELECT
        customer_unique_id,
        COUNT(*) AS order_count
    FROM order_summary
    GROUP BY customer_unique_id
)
SELECT
    COUNT(*) AS total_customers,
    SUM(CASE WHEN order_count > 1 THEN 1 ELSE 0 END)
        AS repeat_customers,
    ROUND(
        100.0 * SUM(CASE WHEN order_count > 1 THEN 1 ELSE 0 END)
        / COUNT(*),
        2
    ) AS repeat_customer_percent
FROM customer_order_counts;


-- Q14: How many days does delivery take on average?
WITH delivered_orders AS (
    SELECT
        order_id,
        delivery_days
    FROM order_summary
    WHERE order_status = 'delivered'
      AND delivery_days IS NOT NULL
)
SELECT
    COUNT(*) AS delivered_orders,
    ROUND(AVG(delivery_days), 2) AS average_delivery_days,
    MIN(delivery_days) AS fastest_delivery_days,
    MAX(delivery_days) AS slowest_delivery_days
FROM delivered_orders;


-- Q15: Which states have the highest late-delivery rate?
WITH state_delivery AS (
    SELECT
        customer_state,
        COUNT(*) AS delivered_orders,
        SUM(late_flag) AS late_orders
    FROM order_summary
    WHERE order_status = 'delivered'
      AND delivery_days IS NOT NULL
    GROUP BY customer_state
)
SELECT
    customer_state,
    delivered_orders,
    late_orders,
    ROUND(
        100.0 * late_orders / delivered_orders,
        2
    ) AS late_delivery_percent
FROM state_delivery
WHERE delivered_orders >= 100
ORDER BY late_delivery_percent DESC;


-- Q16: Which categories have the highest freight costs?
WITH category_freight AS (
    SELECT
        category,
        SUM(freight_value) AS total_freight,
        SUM(item_revenue) AS total_revenue
    FROM fact_sales
    GROUP BY category
)
SELECT
    category,
    ROUND(total_freight, 2) AS total_freight,
    ROUND(total_revenue, 2) AS total_revenue,
    ROUND(
        100.0 * total_freight / NULLIF(total_revenue, 0),
        2
    ) AS freight_percent_of_revenue
FROM category_freight
ORDER BY total_freight DESC
LIMIT 15;


-- Q17: Which seller states generate the most revenue?
WITH seller_state_revenue AS (
    SELECT
        s.seller_state,
        SUM(fs.item_revenue) AS revenue,
        COUNT(DISTINCT fs.order_id) AS order_count
    FROM fact_sales fs
    LEFT JOIN sellers s
        ON fs.seller_id = s.seller_id
    GROUP BY s.seller_state
)
SELECT
    seller_state,
    ROUND(revenue, 2) AS revenue,
    order_count
FROM seller_state_revenue
ORDER BY revenue DESC;


-- Q18: How do monthly customers and orders change?
WITH monthly_activity AS (
    SELECT
        order_month,
        COUNT(DISTINCT order_id) AS order_count,
        COUNT(DISTINCT customer_unique_id) AS customer_count,
        SUM(item_revenue) AS revenue
    FROM fact_sales
    GROUP BY order_month
)
SELECT
    order_month,
    order_count,
    customer_count,
    ROUND(revenue, 2) AS revenue
FROM monthly_activity
ORDER BY order_month;


-- Q19: What revenue comes from each order status?
WITH status_metrics AS (
    SELECT
        order_status,
        COUNT(*) AS order_count,
        SUM(order_revenue) AS revenue
    FROM order_summary
    GROUP BY order_status
)
SELECT
    order_status,
    order_count,
    ROUND(revenue, 2) AS revenue
FROM status_metrics
ORDER BY revenue DESC;


-- Q20: Which states have the best review scores?
WITH review_by_order AS (
    SELECT
        order_id,
        AVG(TRY_CAST(review_score AS DOUBLE)) AS review_score
    FROM reviews
    GROUP BY order_id
),
state_reviews AS (
    SELECT
        os.customer_state,
        AVG(r.review_score) AS average_review_score,
        COUNT(*) AS reviewed_orders
    FROM order_summary os
    INNER JOIN review_by_order r
        ON os.order_id = r.order_id
    GROUP BY os.customer_state
)
SELECT
    customer_state,
    ROUND(average_review_score, 2) AS average_review_score,
    reviewed_orders
FROM state_reviews
WHERE reviewed_orders >= 100
ORDER BY average_review_score DESC;


-- Q21: What is month-over-month revenue growth?
WITH monthly_revenue AS (
    SELECT
        order_month,
        SUM(item_revenue) AS revenue
    FROM fact_sales
    GROUP BY order_month
),
previous_month AS (
    SELECT
        order_month,
        revenue,
        LAG(revenue) OVER (
            ORDER BY order_month
        ) AS previous_revenue
    FROM monthly_revenue
)
SELECT
    order_month,
    ROUND(revenue, 2) AS revenue,
    ROUND(previous_revenue, 2) AS previous_revenue,
    ROUND(
        100.0 * (revenue - previous_revenue)
        / NULLIF(previous_revenue, 0),
        2
    ) AS mom_growth_percent
FROM previous_month
ORDER BY order_month;


-- Q22: What is the cumulative revenue over time?
WITH monthly_revenue AS (
    SELECT
        order_month,
        SUM(item_revenue) AS revenue
    FROM fact_sales
    GROUP BY order_month
)
SELECT
    order_month,
    ROUND(revenue, 2) AS revenue,
    ROUND(
        SUM(revenue) OVER (
            ORDER BY order_month
            ROWS BETWEEN UNBOUNDED PRECEDING
            AND CURRENT ROW
        ),
        2
    ) AS running_revenue
FROM monthly_revenue
ORDER BY order_month;


-- Q23: What are the top three categories in each month?
WITH monthly_category_revenue AS (
    SELECT
        order_month,
        category,
        SUM(item_revenue) AS revenue
    FROM fact_sales
    GROUP BY order_month, category
),
ranked_categories AS (
    SELECT
        order_month,
        category,
        revenue,
        ROW_NUMBER() OVER (
            PARTITION BY order_month
            ORDER BY revenue DESC
        ) AS category_rank
    FROM monthly_category_revenue
)
SELECT
    order_month,
    category,
    ROUND(revenue, 2) AS revenue,
    category_rank
FROM ranked_categories
WHERE category_rank <= 3
ORDER BY order_month, category_rank;


-- Q24: What is the product rank within each category?
WITH product_revenue AS (
    SELECT
        category,
        product_id,
        SUM(item_revenue) AS revenue
    FROM fact_sales
    GROUP BY category, product_id
)
SELECT
    category,
    product_id,
    ROUND(revenue, 2) AS revenue,
    RANK() OVER (
        PARTITION BY category
        ORDER BY revenue DESC
    ) AS product_rank
FROM product_revenue
ORDER BY category, product_rank;


-- Q25: What is the three-month moving-average revenue?
WITH monthly_revenue AS (
    SELECT
        order_month,
        SUM(item_revenue) AS revenue
    FROM fact_sales
    GROUP BY order_month
)
SELECT
    order_month,
    ROUND(revenue, 2) AS revenue,
    ROUND(
        AVG(revenue) OVER (
            ORDER BY order_month
            ROWS BETWEEN 2 PRECEDING AND CURRENT ROW
        ),
        2
    ) AS three_month_moving_average
FROM monthly_revenue
ORDER BY order_month;


-- Q26: What is each customer's order sequence?
SELECT
    customer_unique_id,
    order_id,
    order_date,
    ROW_NUMBER() OVER (
        PARTITION BY customer_unique_id
        ORDER BY order_date
    ) AS customer_order_number
FROM order_summary
ORDER BY customer_unique_id, order_date;


-- Q27: How many days passed between customer orders?
WITH customer_orders AS (
    SELECT
        customer_unique_id,
        order_id,
        order_date,
        LAG(order_date) OVER (
            PARTITION BY customer_unique_id
            ORDER BY order_date
        ) AS previous_order_date
    FROM order_summary
)
SELECT
    customer_unique_id,
    order_id,
    order_date,
    previous_order_date,
    DATE_DIFF(
        'day',
        previous_order_date,
        order_date
    ) AS days_between_orders
FROM customer_orders
WHERE previous_order_date IS NOT NULL
ORDER BY customer_unique_id, order_date;

Writing salespulse_queries.sql


In [17]:
from pathlib import Path

sql_text = Path("salespulse_queries.sql").read_text()

print("SQL file created.")
print("Documented queries:", sql_text.count("-- Q"))

SQL file created.
Documented queries: 27


In [18]:
import os

os.makedirs("exports", exist_ok=True)

con.execute("""
COPY (
    SELECT
        order_id,
        customer_unique_id,
        order_date,
        order_month,
        customer_state AS region,
        category,
        product_id,
        seller_id,
        item_revenue AS revenue,
        freight_value,
        delivery_days,
        late_flag,
        order_status
    FROM fact_sales
)
TO 'exports/salespulse_dashboard.csv'
(
    HEADER,
    DELIMITER ','
)
""")

print("Dashboard CSV created successfully.")

Dashboard CSV created successfully.


In [19]:
dashboard_check = con.execute("""
    SELECT
        COUNT(*) AS dashboard_rows,
        COUNT(DISTINCT order_id) AS distinct_orders,
        COUNT(DISTINCT customer_unique_id) AS distinct_customers,
        ROUND(SUM(revenue), 2) AS total_revenue
    FROM read_csv_auto('exports/salespulse_dashboard.csv')
""").df()

display(dashboard_check)

,dashboard_rows,distinct_orders,distinct_customers,total_revenue
0,113425,99441,96096,13591643.7


In [20]:
from google.colab import files

files.download("salespulse_queries.sql")
files.download("exports/salespulse_dashboard.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [21]:
total_revenue = con.execute("""
    SELECT SUM(item_revenue)
    FROM fact_sales
""").fetchone()[0]

total_orders = con.execute("""
    SELECT COUNT(*)
    FROM order_summary
""").fetchone()[0]

unique_customers = con.execute("""
    SELECT COUNT(DISTINCT customer_unique_id)
    FROM order_summary
""").fetchone()[0]

top_category = con.execute("""
    SELECT
        category,
        SUM(item_revenue) AS revenue
    FROM fact_sales
    WHERE category <> 'unknown'
    GROUP BY category
    ORDER BY revenue DESC
    LIMIT 1
""").fetchone()

top_region = con.execute("""
    SELECT
        customer_state,
        SUM(order_revenue) AS revenue
    FROM order_summary
    GROUP BY customer_state
    ORDER BY revenue DESC
    LIMIT 1
""").fetchone()

top_month = con.execute("""
    SELECT
        order_month,
        SUM(item_revenue) AS revenue
    FROM fact_sales
    GROUP BY order_month
    ORDER BY revenue DESC
    LIMIT 1
""").fetchone()

late_rate = con.execute("""
    SELECT
        100.0 * SUM(late_flag) / COUNT(*)
    FROM order_summary
    WHERE order_status = 'delivered'
      AND delivery_days IS NOT NULL
""").fetchone()[0]

repeat_rate = con.execute("""
    WITH customer_orders AS (
        SELECT
            customer_unique_id,
            COUNT(*) AS order_count
        FROM order_summary
        GROUP BY customer_unique_id
    )
    SELECT
        100.0 * SUM(
            CASE WHEN order_count > 1 THEN 1 ELSE 0 END
        ) / COUNT(*)
    FROM customer_orders
""").fetchone()[0]

print("SALES PULSE EXECUTIVE FINDINGS")
print("--------------------------------")
print(
    f"1. Total revenue was {total_revenue:,.2f} "
    f"across {total_orders:,} orders."
)
print(
    f"2. The company served {unique_customers:,} unique customers."
)
print(
    f"3. The top product category was "
    f"{top_category[0]} with revenue of {top_category[1]:,.2f}."
)
print(
    f"4. The highest-revenue customer region was "
    f"{top_region[0]} with revenue of {top_region[1]:,.2f}."
)
print(
    f"5. The highest-revenue month was "
    f"{top_month[0]} with revenue of {top_month[1]:,.2f}."
)
print(
    f"6. The late-delivery rate among delivered orders was "
    f"{late_rate:.2f}%."
)
print(
    f"7. The repeat-customer rate was "
    f"{repeat_rate:.2f}%."
)

SALES PULSE EXECUTIVE FINDINGS
--------------------------------
1. Total revenue was 13,591,643.70 across 99,441 orders.
2. The company served 96,096 unique customers.
3. The top product category was health_beauty with revenue of 1,258,681.34.
4. The highest-revenue customer region was SP with revenue of 5,202,955.05.
5. The highest-revenue month was 2017-11-01 with revenue of 1,010,271.37.
6. The late-delivery rate among delivered orders was 8.11%.
7. The repeat-customer rate was 3.12%.
